<a href="https://colab.research.google.com/github/setdaygamer12/B-i-t-p-week-3/blob/app-d%E1%BB%B1-%C4%91o%C3%A1n-ti%E1%BB%81n-%C4%91i%E1%BB%87n/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas scikit-learn streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.8 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import re
import time

@st.cache_data
def load_data():
    try:
        df = pd.read_csv('/content/Khao_sat_tien_dien_phong_tro.csv')
        house_col = df.columns[4]
        price_col = df.columns[10]

        df[house_col] = df[house_col].apply(lambda x: re.sub(r'( bình thường)+', ' bình thường', str(x)).strip())
        avg_prices = df.groupby(house_col)[price_col].mean().to_dict()
        return avg_prices
    except Exception as e:
        return {"Phòng trọ bình thường": 3800, "Chung cư": 3000, "Ký túc xá": 3500, "Nhà nguyên căn": 3000}

avg_prices = load_data()
st.set_page_config(page_title="AI Electricity Master", page_icon="⚡", layout="wide")

st.markdown("""
    <style>
    .stApp { background-color: #0E1117; color: #FFFFFF; }
    label, .stMarkdown p { color: #E0E6ED !important; font-weight: 500 !important; }
    div[role="radiogroup"] label p { color: #FFFFFF !important; }

    .title-text {
        background: linear-gradient(90deg, #FFD700, #FFFFFF);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-family: 'Segoe UI', sans-serif;
        font-weight: 800;
        text-align: center;
        text-transform: uppercase;
        letter-spacing: 3px;
        margin-bottom: 30px;
    }

    div.stButton > button:first-child {
        background: linear-gradient(135deg, #FFD700 0%, #F0B232 100%);
        color: #000000 !important;
        border: none;
        padding: 18px;
        font-size: 22px;
        font-weight: 800;
        border-radius: 15px;
        box-shadow: 0 0 20px rgba(255, 215, 0, 0.4);
    }

    .main-card {
        background-color: #1C2128;
        padding: 35px;
        border-radius: 25px;
        border: 2px solid #30363D;
        border-top: 10px solid #FFD700;
        text-align: center;
    }

    .card-label { color: #FFD700 !important; font-weight: bold; letter-spacing: 2px; font-size: 1.2rem; }
    .price-tag { font-size: 4.5rem; font-weight: 900; color: #4ADE80; text-shadow: 0px 0px 25px rgba(74, 222, 128, 0.5); }
    [data-testid="stMetricValue"] { color: #FFD700 !important; }
    </style>
    """, unsafe_allow_html=True)

st.markdown('<h1 class="title-text">⚡ HỆ THỐNG DỰ TOÁN TIỀN ĐIỆN</h1>', unsafe_allow_html=True)

col_input, col_result = st.columns([1, 1.2], gap="large")

with col_input:
    st.markdown("### 📋 THÔNG TIN NHẬP LIỆU")
    c1, c2 = st.columns(2)
    with c1:
        num_people = st.number_input("👥 SỐ NGƯỜI Ở", 1, 15, 2)
        dien_tich = st.number_input("📏 DIỆN TÍCH (M2)", 10, 150, 25)
    with c2:
        house_type = st.selectbox("🏠 LOẠI HÌNH NƠI Ở", options=list(avg_prices.keys()))
        lau = st.number_input("🏢 TẦNG LẦU", 0, 10, 0)

    st.markdown("---")
    st.markdown("### ⚙️ CẤU HÌNH THIẾT BỊ")
    num_ac = st.slider("Số lượng máy lạnh", 0, 5, 1)
    ac_hours = st.slider("⏱️ Thời gian sử dụng máy lạnh (giờ/ngày)", 0, 24, 8)

    c3, c4 = st.columns(2)
    with c3:
        num_fans = st.number_input("🌬️ SỐ LƯỢNG QUẠT", 0, 10, 2)
    with c4:
        has_fridge_text = st.radio("🧊 CÓ TỦ LẠNH KHÔNG?", ["Có", "Không"], horizontal=True)

    has_fridge = True if has_fridge_text == "Có" else False

    st.write(" ")
    predict_btn = st.button("🚀 XÁC NHẬN DỰ TOÁN")

with col_result:
    if predict_btn:
        with st.spinner('Đang phân tích dữ liệu thực tế...'):
            time.sleep(0.6)
            unit_price = avg_prices.get(house_type, 3500)

            base_kwh = num_people * 22
            fridge_kwh = 30 if has_fridge else 0
            fan_kwh = num_fans * 15
            ac_kwh = num_ac * 0.85 * ac_hours * 30 * 0.7

            total_kwh = base_kwh + fridge_kwh + fan_kwh + ac_kwh
            total_money = total_kwh * unit_price

            st.markdown(f"""
                <div class="main-card">
                    <p class="card-label">CHI PHÍ DỰ KIẾN TRONG THÁNG</p>
                    <h1 class="price-tag">{total_money:,.0f} <span style="font-size: 2rem">VNĐ</span></h1>
                    <p style="color: #FFFFFF; font-style: italic;">Dựa trên đơn giá {unit_price:,.0f}đ/kWh ({house_type})</p>
                </div>
                """, unsafe_allow_html=True)

            st.write(" ")
            m1, m2, m3 = st.columns(3)
            m1.metric("Tiêu thụ", f"{total_kwh:.1f} kWh")
            m2.metric("Điều hòa", f"{ac_hours}h/ngày")
            m3.metric("Thiết bị", f"{num_ac + num_fans + (1 if has_fridge else 0)}")

            st.markdown("### 💡 LỜI KHUYÊN TIẾT KIỆM")
            if ac_hours >= 10:
                st.error("❗ **Máy lạnh:** Bạn dùng >10h. Hãy kiểm tra độ kín của cửa để tránh thoát nhiệt.")
            elif ac_hours > 0:
                st.info("❄️ **Mẹo:** Để máy lạnh ở 26°C kèm quạt máy sẽ giúp tiết kiệm điện hơn để 20°C.")

            if lau >= 3:
                st.warning("☀️ **Vị trí:** Tầng cao thường nóng hơn. Hãy dùng rèm cửa dày để cản nắng.")

            st.success("🔌 **Lưu ý:** Rút phích cắm các thiết bị điện khi không sử dụng để giảm 'điện chờ'.")

    else:
        st.markdown("""
            <div style="text-align: center; padding: 50px; border: 2px dashed #30363D; border-radius: 20px;">
                <h2 style="color: #FFD700;">CHỜ NHẬP LIỆU...</h2>
                <p style="color: #E0E6ED;">Hoàn thành thông tin bên trái và nhấn nút Xác Nhận.</p>
            </div>
            """, unsafe_allow_html=True)

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import os

CONF_TOKEN = "3DM8HMqQBF9DAXw6ndnyjcLVsZC_7dVwieTvfpVW1TVPzGqsr"
!ngrok config add-authtoken {CONF_TOKEN}
ngrok.kill()
os.system("streamlit run app.py &")
public_url = ngrok.connect(8501).public_url
print(f"\n🚀 Ứng dụng đã online!")
print(f"👉 Truy cập tại đây: {public_url}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

🚀 Ứng dụng đã online!
👉 Truy cập tại đây: https://slander-plow-trustee.ngrok-free.dev
